# Chapter 13 - Berquist-Sherman Techniques


In [ ]:
import numpy as np
import pandas as pd
import chainladder as cl
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Helper functions, skip to the next section for actual exhibits
def as_series(tri):
    s = tri.to_frame(origin_as_datetime=False).iloc[:, 0]
    s.index = [int(getattr(i, "year", i)) for i in s.index]
    return s


def triangle_frame(tri):
    """Origin x development frame with integer accident-year index."""
    df = tri.to_frame(origin_as_datetime=False)
    df.index = [int(getattr(i, "year", i)) for i in df.index]
    return df


def link_ratio_frame(tri, decimals=3):
    df = np.round(tri.link_ratio.to_frame(origin_as_datetime=False), decimals)
    df.index = [int(getattr(i, "year", i)) for i in df.index]
    return df


## P294 (Exhibit I Sheet 1)


In [ ]:
mm = cl.load_sample("friedland_med_mal")
reported = mm["Reported Claims"]

# PART 1 — Unadjusted reported claims triangle
exhibit_i_s1_tri = triangle_frame(reported)
display(exhibit_i_s1_tri)

# PART 2 — Age-to-age factors
exhibit_i_s1_lr = link_ratio_frame(reported)
display(exhibit_i_s1_lr)

# PART 3 — Simple average of all years
dev_reported = cl.Development(average="simple", n_periods=-1).fit(reported)
avg_ldf = np.round(dev_reported.ldf_.to_frame(origin_as_datetime=False).values.flatten(), 3)
exhibit_i_s1_avg = pd.Series(
    avg_ldf,
    index=["12-24", "24-36", "36-48", "48-60", "60-72", "72-84", "84-96"],
    name="Simple Average All Years",
)
display(exhibit_i_s1_avg.to_frame().T)

# PART 4 — Selected LDFs (simple average + 1.000 tail), CDF, percent reported
selected_ldf = {
    12: 2.532,
    24: 1.921,
    36: 1.503,
    48: 1.170,
    60: 1.206,
    72: 1.052,
    84: 1.027,
    96: 1.000,
}
reported_dev = cl.DevelopmentConstant(patterns=selected_ldf, style="ldf").fit(reported)
cdf = np.round(reported_dev.cdf_.to_frame(origin_as_datetime=False).values.flatten(), 3)
pct_reported = np.round(1.0 / cdf, 3)

exhibit_i_s1_sel = pd.DataFrame(
    {
        "Selected": list(selected_ldf.values()),
        "CDF to Ultimate": cdf,
        "Percent Reported": pct_reported,
    },
    index=["12-24", "24-36", "36-48", "48-60", "60-72", "72-84", "84-96", "96-108"],
)
display(exhibit_i_s1_sel.T)


In [ ]:
# Exhibit I Sheet 1 — reconcile to Friedland PDF p294
assert np.allclose(
    exhibit_i_s1_tri.values,
    [
        [2897000, 5160000, 10714000, 15228000, 16611000, 20899000, 22892000, 23506000],
        [4828000, 10707000, 16907000, 22840000, 26211000, 31970000, 32216000, np.nan],
        [5455000, 11941000, 20733000, 30928000, 42395000, 48377000, np.nan, np.nan],
        [8732000, 18633000, 32143000, 57196000, 61163000, np.nan, np.nan, np.nan],
        [11228000, 19967000, 50143000, 73733000, np.nan, np.nan, np.nan, np.nan],
        [8706000, 33459000, 63477000, np.nan, np.nan, np.nan, np.nan, np.nan],
        [12928000, 48904000, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
        [15791000, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    ],
    equal_nan=True,
)
assert np.allclose(
    exhibit_i_s1_lr.loc[1969].dropna().values,
    [1.781, 2.076, 1.421, 1.091, 1.258, 1.095, 1.027],
)
assert np.allclose(
    exhibit_i_s1_lr.loc[1970].dropna().values,
    [2.218, 1.579, 1.351, 1.148, 1.220, 1.008],
)
assert np.allclose(
    exhibit_i_s1_lr.loc[1971].dropna().values,
    [2.189, 1.736, 1.492, 1.371, 1.141],
)
assert np.allclose(
    exhibit_i_s1_lr.loc[1972].dropna().values,
    [2.134, 1.725, 1.779, 1.069],
)
assert np.allclose(
    exhibit_i_s1_lr.loc[1973].dropna().values,
    [1.778, 2.511, 1.470],
)
assert np.allclose(exhibit_i_s1_lr.loc[1974].dropna().values, [3.843, 1.897])
assert np.allclose(exhibit_i_s1_lr.loc[1975].dropna().values, [3.783])
assert np.allclose(avg_ldf, [2.532, 1.921, 1.503, 1.170, 1.206, 1.052, 1.027])
assert np.allclose(
    exhibit_i_s1_sel["Selected"],
    [2.532, 1.921, 1.503, 1.170, 1.206, 1.052, 1.027, 1.000],
)
assert np.allclose(
    exhibit_i_s1_sel["CDF to Ultimate"],
    [11.145, 4.402, 2.291, 1.524, 1.303, 1.080, 1.027, 1.000],
)
assert np.allclose(
    exhibit_i_s1_sel["Percent Reported"],
    [0.090, 0.227, 0.436, 0.656, 0.767, 0.926, 0.974, 1.000],
)
